In [ ]:
!pip uninstall -y ultralytics
!pip install -U ultralytics huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 59.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0


In [ ]:
# Install the latest Ultralytics (which includes SAM 3) and Hugging Face
!pip install -U ultralytics huggingface_hub

# Create the frames folder
!mkdir -p /content/frames

# Extract the video into individual JPEGs
!ffmpeg -i /stephcurryvideo.mp4 /content/frames/%05d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from huggingface_hub import login

# Your specific Hugging Face access token
login(token="ADD_YOUR_TOKEN_AND_KEEP_IT_SAVED")

In [ ]:
!pip install -U ultralytics huggingface_hub
!mkdir -p /content/frames
!ffmpeg -i /content/stephcurryvideo.mp4 /content/frames/%05d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from huggingface_hub import login, hf_hub_download

token = "ADD_YOUR_TOKEN_AND_KEEP_IT_SAVED"
login(token=token)

print("Downloading SAM 3 weights from Hugging Face... This may take a minute or two.")
hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt",
    local_dir="/content",
    token=token
)
print("✅ Weights downloaded successfully.")

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

✅ Weights downloaded successfully.


In [ ]:
import os
import json
import torch
import logging
import numpy as np
from ultralytics import SAM

# 1. Silence Ultralytics logging to keep the output completely clean
logging.getLogger("ultralytics").setLevel(logging.ERROR)

# 2. Configuration
JSON_PATH = '/tracking_data.json'
FRAMES_DIR = "/content/frames"
OUTPUT_MASK_DIR = "/content/masks"
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
%cd /content

# 3. Clear out VRAM
torch.cuda.empty_cache()

# 4. Load SAM 3
print("Loading SAM 3...")
model = SAM("sam3.pt")

# 5. Load Tracking Data
with open(JSON_PATH, 'r') as f:
    tracking_data = json.load(f)

frame_list = sorted([k for k in tracking_data.keys()])
print(f"Starting SAM 3 segmentation for {len(frame_list)} frames...")

# 6. Process Frame-by-Frame
for i, key in enumerate(frame_list):
    frame_idx = int(key.split('_')[1])
    frame_path = os.path.join(FRAMES_DIR, f"{frame_idx:05d}.jpg")

    if not os.path.exists(frame_path):
        continue

    boxes = [det["box"] for det in tracking_data[key]]
    if not boxes:
        continue

    # FIX: Changed quantize=True to quantize='fp16'
    results = model(frame_path, bboxes=boxes, device=0, quantize='fp16', imgsz=1036, verbose=False)

    if results[0].masks is not None:
        mask = results[0].masks.data.cpu().numpy()
        np.save(os.path.join(OUTPUT_MASK_DIR, f"mask_{frame_idx:05d}.npy"), mask)

    # Print progress update every 50 frames
    if i % 50 == 0 and i > 0:
        print(f"Processed {i}/{len(frame_list)} frames...")

print("✅ SAM 3 Segmentation complete! All masks are saved in /content/masks.")

/content
Loading SAM 3...
Starting SAM 3 segmentation for 504 frames...
Processed 50/504 frames...
Processed 100/504 frames...
Processed 150/504 frames...
Processed 200/504 frames...
Processed 250/504 frames...
Processed 300/504 frames...
Processed 350/504 frames...
Processed 400/504 frames...
Processed 450/504 frames...


KeyboardInterrupt: 

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# 1. Configuration Paths
FRAMES_DIR = "/content/frames"
MASKS_DIR = "/content/masks"
OUTPUT_VIDEO_PATH = "/content/tracked_output.mp4"

# 2. Get sorted list of frames
frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
if not frame_files:
    raise FileNotFoundError("No frames found in /content/frames")

# 3. Read first frame to get dimensions
sample_img = cv2.imread(os.path.join(FRAMES_DIR, frame_files[0]))
height, width, layers = sample_img.shape

# 4. Initialize Video Writer (MP4V codec for high Colab compatibility)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, 30.0, (width, height))

print(f"Compiling tracking video from {len(frame_files)} frames...")

# 5. Loop through frames and overlay masks
for frame_file in tqdm(frame_files):
    frame_idx = int(frame_file.split('.')[0])
    frame_path = os.path.join(FRAMES_DIR, frame_file)
    mask_path = os.path.join(MASKS_DIR, f"mask_{frame_idx:05d}.npy")

    img = cv2.imread(frame_path)

    # If a mask exists for this frame, overlay it
    if os.path.exists(mask_path):
        mask = np.load(mask_path)

        # SAM outputs might have an extra batch/channel dimension (e.g., [1, H, W] or [H, W])
        if mask.ndim == 3:
            mask = mask[0]

        # Ensure mask is binary boolean array
        mask = mask.astype(bool)

        # Create a colored highlight overlay (BGR format - this creates a bright neon blue highlight)
        overlay_color = [255, 128, 0]

        # Create a copy of the image to apply the blend
        overlay_img = img.copy()
        overlay_img[mask] = overlay_color

        # Blend the original image and the colored overlay (alpha=0.6 makes it semi-transparent)
        alpha = 0.6
        img = cv2.addWeighted(overlay_img, alpha, img, 1 - alpha, 0)

    # Write the frame to the video sequence
    video.write(img)

# Clean up
video.release()
print(f"\n✅ Video compilation complete! Saved to: {OUTPUT_VIDEO_PATH}")

Compiling tracking video from 504 frames...


100%|██████████| 504/504 [00:30<00:00, 16.59it/s]


✅ Video compilation complete! Saved to: /content/tracked_output.mp4


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

FRAMES_DIR = "/content/frames"
MASKS_DIR = "/content/masks"
OUTPUT_VIDEO_PATH = "/content/all_players_tracked.mp4"

frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
sample_img = cv2.imread(os.path.join(FRAMES_DIR, frame_files[0]))
height, width, _ = sample_img.shape

video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (width, height))
print("Compiling video with ALL masks...")

for frame_file in tqdm(frame_files):
    frame_idx = int(frame_file.split('.')[0])
    img = cv2.imread(os.path.join(FRAMES_DIR, frame_file))
    mask_path = os.path.join(MASKS_DIR, f"mask_{frame_idx:05d}.npy")

    if os.path.exists(mask_path):
        masks = np.load(mask_path) # Load all masks for this frame

        # Handle different output shapes from SAM
        if masks.ndim == 4: masks = masks[0]
        elif masks.ndim == 2: masks = np.expand_dims(masks, 0)

        # Combine all player masks into one single master mask
        combined_mask = np.any(masks, axis=0)

        overlay_img = img.copy()
        overlay_img[combined_mask] = [255, 128, 0] # Neon Blue overlay
        img = cv2.addWeighted(overlay_img, 0.6, img, 0.4, 0)

    video.write(img)

video.release()
print(f"✅ Video complete! Saved to: {OUTPUT_VIDEO_PATH}")

Compiling video with ALL masks...


100%|██████████| 504/504 [00:32<00:00, 15.43it/s]

✅ Video complete! Saved to: /content/all_players_tracked.mp4


In [ ]:
!pip install -q rfdetr supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 12.0 MB/s eta 0:00:00


In [ ]:
!pip install -q rfdetr supervision pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.4 MB/s eta 0:00:00


In [ ]:
!pip install -q rfdetr supervision pytorch-lightning faster-coco-eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 11.3 MB/s eta 0:00:00


In [ ]:
!mkdir -p /content/frames
!ffmpeg -i /stephcurryvideo.mp4 /content/frames/%05d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
!pip install -q rfdetr supervision pytorch-lightning faster-coco-eval

In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm
from rfdetr import RFDETR

# 1. Your exact file path
MODEL_PATH = '/checkpoint_best_ema.pth'

if not os.path.exists(MODEL_PATH) and os.path.exists('/content/checkpoint_best_ema.pth'):
    MODEL_PATH = '/content/checkpoint_best_ema.pth'

FRAMES_DIR = '/content/frames'
OUTPUT_JSON_PATH = '/content/custom_tracking_data.json'

print(f"Loading custom RF-DETR weights from: {MODEL_PATH}")

# 2. Load the model using the smart wrapper
model = RFDETR.from_checkpoint(MODEL_PATH)

# Optional: optimize for inference if supported
if hasattr(model, 'optimize_for_inference'):
    model.optimize_for_inference()

# 3. Setup Frame Processing
frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
tracking_data = {}

print("Detecting players and ignoring the crowd...")
for frame_file in tqdm(frame_files):
    frame_idx = int(frame_file.split('.')[0])
    frame_key = f"frame_{frame_idx}"

    frame_path = os.path.join(FRAMES_DIR, frame_file)
    img = Image.open(frame_path).convert("RGB")

    # 4. Predict using your custom weights
    detections = model.predict(img, threshold=0.5)

    tracking_data[frame_key] = []

    # 5. Extract bounding boxes for SAM 3
    if detections is not None and len(detections) > 0:
        for box in detections.xyxy:
            tracking_data[frame_key].append({
                "box": [float(box[0]), float(box[1]), float(box[2]), float(box[3])],
                "id": 1 # Dummy ID since your model only detects players
            })

# 6. Save the data
with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(tracking_data, f, indent=4)

print(f"\n✅ Success! Clean player data saved to {OUTPUT_JSON_PATH}")

Loading custom RF-DETR weights from: /checkpoint_best_ema.pth


[2026-06-28 10:20:33] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-28 10:20:33] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-28 10:20:35] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Detecting players and ignoring the crowd...


100%|██████████| 504/504 [00:39<00:00, 12.73it/s]


✅ Success! Clean player data saved to /content/custom_tracking_data.json


In [ ]:
import os
import json
import torch
import logging
import numpy as np
from huggingface_hub import login, hf_hub_download
from ultralytics import SAM

# 1. Environment and Login Setup
# (Make sure you have run: !pip install -q -U ultralytics huggingface_hub)
token = "ADD_YOUR_TOKEN_AND_KEEP_IT_SAVED"
login(token=token)

print("Downloading SAM 3 weights...")
hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_dir="/content", token=token)

# 2. Configuration
logging.getLogger("ultralytics").setLevel(logging.ERROR)
JSON_PATH = '/content/custom_tracking_data.json'
FRAMES_DIR = "/content/frames"
OUTPUT_MASK_DIR = "/content/masks"

# Clear old masks if they exist from a previous run
os.system("rm -rf /content/masks")
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
os.chdir('/content')

# 3. Load SAM 3
torch.cuda.empty_cache()
print("Loading SAM 3...")
model = SAM("sam3.pt")

# 4. Load Custom RF-DETR Tracking Data
with open(JSON_PATH, 'r') as f:
    tracking_data = json.load(f)

frame_list = sorted([k for k in tracking_data.keys()])
print(f"Starting SAM 3 segmentation for {len(frame_list)} frames...")

# 5. Process Frame-by-Frame safely
for i, key in enumerate(frame_list):
    frame_idx = int(key.split('_')[1])
    frame_path = os.path.join(FRAMES_DIR, f"{frame_idx:05d}.jpg")

    if not os.path.exists(frame_path):
        continue

    boxes = [det["box"] for det in tracking_data[key]]
    if not boxes:
        continue

    # Memory-safe precision without deprecation warnings
    results = model(frame_path, bboxes=boxes, device=0, quantize='fp16', imgsz=1036, verbose=False)

    if results[0].masks is not None:
        mask = results[0].masks.data.cpu().numpy()
        np.save(os.path.join(OUTPUT_MASK_DIR, f"mask_{frame_idx:05d}.npy"), mask)

    if i % 50 == 0 and i > 0:
        print(f"Processed {i}/{len(frame_list)} frames...")

print("✅ SAM 3 Segmentation complete! All masks are saved in /content/masks.")

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

Loading SAM 3...
Starting SAM 3 segmentation for 504 frames...
Processed 50/504 frames...
Processed 100/504 frames...
Processed 150/504 frames...
Processed 200/504 frames...
Processed 250/504 frames...
Processed 300/504 frames...
Processed 350/504 frames...
Processed 400/504 frames...
Processed 450/504 frames...
Processed 500/504 frames...
✅ SAM 3 Segmentation complete! All masks are saved in /content/masks.


In [ ]:
import os
from google.colab import files

# Zip the entire masks folder into one file
os.system("zip -r /content/masks_backup.zip /content/masks")

# Trigger the browser download
files.download("/content/masks_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# 1. Unzip the backup masks directly into the correct folder
print("📦 Unzipping masks from backup...")
os.system("unzip -o -q /masks_backup.zip -d /")

# 2. Re-extract frames from the video really quickly
print("🎞️ Extracting frames from video...")
os.makedirs("/content/frames", exist_ok=True)
os.system("ffmpeg -y -i /stephcurryvideo.mp4 /content/frames/%05d.jpg -hide_banner -loglevel error")

# 3. Setup paths for video compilation
FRAMES_DIR = "/content/frames"
MASKS_DIR = "/content/masks"
OUTPUT_VIDEO_PATH = "/content/all_players_tracked2.mp4"

# Verify extraction worked
frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
if not frame_files:
    raise Exception("No frames found! Make sure stephcurryvideo.mp4 is uploaded properly.")

# 4. Initialize Video Writer
sample_img = cv2.imread(os.path.join(FRAMES_DIR, frame_files[0]))
height, width, _ = sample_img.shape
video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (width, height))

print("🎬 Compiling video with ALL player masks...")

# 5. Loop through frames and overlay masks
for frame_file in tqdm(frame_files):
    frame_idx = int(frame_file.split('.')[0])
    img = cv2.imread(os.path.join(FRAMES_DIR, frame_file))
    mask_path = os.path.join(MASKS_DIR, f"mask_{frame_idx:05d}.npy")

    if os.path.exists(mask_path):
        masks = np.load(mask_path)

        # Handle different output shapes from SAM
        if masks.ndim == 4:
            masks = masks[0]
        elif masks.ndim == 2:
            masks = np.expand_dims(masks, 0)

        # Combine all player masks into one single master mask
        combined_mask = np.any(masks, axis=0)

        overlay_img = img.copy()
        overlay_img[combined_mask] = [255, 128, 0] # Neon Blue overlay
        img = cv2.addWeighted(overlay_img, 0.6, img, 0.4, 0)

    video.write(img)

video.release()
print(f"\n✅ Video complete! Saved to: {OUTPUT_VIDEO_PATH}")

# 6. Trigger auto-download in Colab
try:
    from google.colab import files
    files.download(OUTPUT_VIDEO_PATH)
except ImportError:
    pass

📦 Unzipping masks from backup...
🎞️ Extracting frames from video...
🎬 Compiling video with ALL player masks...


100%|██████████| 504/504 [00:11<00:00, 42.05it/s]


✅ Video complete! Saved to: /content/all_players_tracked2.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Configuration
FRAMES_DIR = "/content/frames"
MASKS_DIR = "/content/masks"
OUTPUT_VIDEO_PATH = "/content/all_silhouettes.mp4"

# Verify folders exist
if not os.path.exists(FRAMES_DIR) or not os.path.exists(MASKS_DIR):
    raise FileNotFoundError("Make sure the 'frames' and 'masks' folders are in /content/")

frame_files = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
height, width = cv2.imread(os.path.join(FRAMES_DIR, frame_files[0])).shape[:2]
video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (width, height))

print("🎬 Creating silhouette-only video for all players...")

for frame_file in tqdm(frame_files):
    frame_idx = int(frame_file.split('.')[0])
    mask_path = os.path.join(MASKS_DIR, f"mask_{frame_idx:05d}.npy")

    # Create a solid black background
    img = np.zeros((height, width, 3), dtype=np.uint8)

    if os.path.exists(mask_path):
        masks = np.load(mask_path)

        # Combine ALL player masks into one master silhouette mask
        # Handles various SAM output dimensions
        if masks.ndim == 4:
            combined_mask = np.any(masks[0], axis=0)
        elif masks.ndim == 3:
            combined_mask = np.any(masks, axis=0)
        else:
            combined_mask = masks

        # Draw all silhouettes in Neon Orange
        img[combined_mask] = [0, 165, 255]

    video.write(img)

video.release()
print(f"✅ Success! Silhouette video saved to: {OUTPUT_VIDEO_PATH}")

# Trigger download
from google.colab import files
files.download(OUTPUT_VIDEO_PATH)

🎬 Creating silhouette-only video for all players...


100%|██████████| 504/504 [00:06<00:00, 73.39it/s]

✅ Success! Silhouette video saved to: /content/all_silhouettes.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>